In [ ]:
import folium
import pandas as pd
import geopandas as gpd
from glob import glob
from os.path import join
import branca.colormap as cm
from folium.plugins import TimeSliderChoropleth

In [ ]:
data_path = ""
homeloc_zips_path = ""
path_zips = ""

In [ ]:
daily_files = sorted(glob(join(data_path, "veraset*")))
gdf_zips = gpd.read_file(path_zips)
gdf_zips.set_index('ZIP_CODE', inplace=True)
gdf_zips.drop('OBJECTID', axis=1, inplace=True)
homeloc_zips = pd.read_parquet(homeloc_zips_path, engine='pyarrow')
homeloc_zips.set_index('uid', inplace=True)
homeloc_zips.drop(['lat', 'lng'], axis=1, inplace=True)

In [ ]:
datetime_index = pd.date_range(daily_files[0].split("veraset_")[1].split("_PST")[0],
                               daily_files[-1].split("veraset_")[1].split("_PST")[0])
dt_index_epochs = datetime_index.astype("int64") // 10 ** 9
dt_index = dt_index_epochs.astype("U10")

In [ ]:
df_color = pd.DataFrame(index=gdf_zips.index, columns=dt_index)
df_opacity = pd.DataFrame(index=gdf_zips.index, columns=dt_index)
for file in daily_files:
    date = file.split("veraset_")[1].split("_PST")[0]
    date = str(pd.to_datetime(date, format="%Y-%m-%d").value // 10 ** 9)
    data_f = pd.read_parquet(join(data_path, file), engine='pyarrow')
    data_f.set_index('uid', inplace=True)
    data_f = data_f.join(homeloc_zips, how='left') #find zips from lat/lng
    df_color.loc[:, date] = data_f.groupby("zipcode").sum().astype("float64")
    df_opacity.loc[:, date] = data_f.groupby("zipcode").size().astype("int64")

min_color = df_color.min().min()
max_color = df_color.max().max()
cmap = cm.linear.PuRd_09.scale(min_color, max_color)
color = df_color.map(cmap, na_action="ignore")
color = color.fillna(0, downcast="int64")
color = color.set_index(pd.Index(["color"]*color.shape[0]), append=True)

opacity_min = df_opacity.min().min()
opacity_max = df_opacity.max().max()
opacity = (df_opacity - opacity_min) / (opacity_max - opacity_min)
opacity = opacity.fillna(0, downcast="int64")
opacity = opacity.set_index(pd.Index(["opacity"]*opacity.shape[0]), append=True)

df_style = pd.concat([color.unstack(1), opacity.unstack(1)], axis=1).stack(level=[0])

In [ ]:
styledict = {}
for zip in gdf_zips.index:
    styledict[zip] = df_style.loc[zip].to_dict(orient="index")

In [ ]:
m = folium.Map(location=[34.0549, -118.2426],
               zoom_start=8)

TimeSliderChoropleth(
    gdf_zips.to_json(),
    styledict=styledict,
).add_to(m)

m

![title](img/hl_byzip_bytime.png)

In [ ]:
m.save("home_locations_byzip_bytime.html")